In [1]:
import re
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
import pandas as pd


BASE_URL = "https://scouttunisien.com"
LIST_URL = "https://scouttunisien.com/partenariats/"
OUTPUT_XLSX = "partenariats_projects.xlsx"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
    )
}


def fetch_soup(url: str) -> BeautifulSoup:
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def norm_text(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()


def first_text(el) -> str:
    if not el:
        return ""
    return norm_text(" ".join(el.stripped_strings))


def find_read_more(article) -> str | None:
    # 1) Elementor read more
    a = article.select_one("a.elementor-post__read-more")
    if a and a.get("href"):
        return a["href"]

    # 2) Title link
    a = article.select_one(".elementor-post__title a, h2 a, h3 a")
    if a and a.get("href"):
        return a["href"]

    # 3) Any anchor with "voir" / "see" in text
    for a in article.select("a[href]"):
        t = first_text(a).lower()
        if "voir" in t or "see" in t or "more" in t or "plus" in t:
            return a.get("href")

    # 4) Fallback: first anchor
    a = article.find("a", href=True)
    return a["href"] if a else None


def extract_title(soup: BeautifulSoup) -> str:
    sel = [
        "h1.entry-title",
        ".elementor-heading-title",
        "article h1",
        "header h1",
    ]
    for s in sel:
        el = soup.select_one(s)
        if el:
            return first_text(el)

    # Meta fallback
    og = soup.find("meta", attrs={"property": "og:title"})
    if og and og.get("content"):
        return norm_text(og["content"])

    title_tag = soup.find("title")
    return norm_text(title_tag.get_text()) if title_tag else ""


def extract_date(soup: BeautifulSoup) -> str:
    time_el = soup.select_one("time[datetime]")
    if time_el and time_el.get("datetime"):
        return norm_text(time_el["datetime"])
    if time_el:
        return first_text(time_el)

    el = soup.select_one(".elementor-post-date, .posted-on time, .entry-meta time")
    if el:
        dt = el.get("datetime")
        return norm_text(dt or first_text(el))

    meta_pub = soup.find("meta", attrs={"property": "article:published_time"})
    if meta_pub and meta_pub.get("content"):
        return norm_text(meta_pub["content"])

    meta_date = soup.find("meta", attrs={"name": "date"})
    if meta_date and meta_date.get("content"):
        return norm_text(meta_date["content"])

    return ""


def extract_description(soup: BeautifulSoup) -> str:
    for s in [
        ".entry-content",
        "article .entry-content",
        "main .entry-content",
        ".elementor-widget-text-editor",
        "article .elementor-widget-text-editor",
        "main .elementor-widget-text-editor",
    ]:
        el = soup.select_one(s)
        if el:
            txt = first_text(el)
            if txt:
                return txt

    meta_desc = soup.find("meta", attrs={"name": "description"})
    if meta_desc and meta_desc.get("content"):
        return norm_text(meta_desc["content"])

    og_desc = soup.find("meta", attrs={"property": "og:description"})
    if og_desc and og_desc.get("content"):
        return norm_text(og_desc["content"])

    main = soup.select_one("main, article")
    return first_text(main) if main else ""


def scrape_partenariats(list_url: str = LIST_URL) -> pd.DataFrame:
    list_soup = fetch_soup(list_url)

    articles = list_soup.select(".elementor-posts-container article")
    if not articles:
        articles = list_soup.select("article")

    results = []
    seen_urls = set()

    for art in articles:
        href = find_read_more(art)
        if not href:
            continue
        detail_url = urljoin(BASE_URL, href)
        if detail_url in seen_urls:
            continue
        seen_urls.add(detail_url)

        try:
            detail_soup = fetch_soup(detail_url)
        except Exception as e:
            print(f"Failed to fetch detail {detail_url}: {e}")
            continue

        title = extract_title(detail_soup)
        date = extract_date(detail_soup)
        desc = extract_description(detail_soup)

        results.append(
            {
                "project_name": title,
                "date": date,
                "description": desc,
                "url": detail_url,
            }
        )

    return pd.DataFrame(results)


def save_to_excel(df: pd.DataFrame, path: str = OUTPUT_XLSX):
    # Keep only requested columns and rename project_name -> name
    out = (
        df[["project_name", "date", "description"]]
        .rename(columns={"project_name": "name"})
    )
    # Requires 'openpyxl' installed
    out.to_excel(path, index=False)
    print(f"Saved {len(out)} rows to {path}")


if __name__ == "__main__":
    df = scrape_partenariats()
    print("rows:", len(df))
    if df.empty:
        print("No projects found.")
    else:
        save_to_excel(df, OUTPUT_XLSX)
        # Optional: preview first few rows
        print(df[["project_name", "date", "description"]].head(10))

rows: 5
Saved 5 rows to partenariats_projects.xlsx
                            project_name                       date  \
0                         L’atelier Rose  2023-06-01T17:21:09+00:00   
1                 Atelier bleu Sfax V1.0  2023-06-01T17:20:34+00:00   
2            Atelier bleu Kerkennah V2.0  2023-06-01T17:19:12+00:00   
3  Projet National Solidarité (Covid-19)  2023-06-01T17:08:49+00:00   
4                        Projet Madrasti  2023-06-01T15:28:01+00:00   

                                         description  
0  le projet porte sue la sensibilisation des jeu...  
1  le projet vise à encadrer la population locale...  
2  le projet porte sur ​La Sensibilisation des fe...  
3  310 Interventions de désinfections et de sensi...  
4  Instauration de 35 Jardins écologiques au sein...  


In [2]:
import re
import time
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
import pandas as pd


BASE_URL = "https://scouttunisien.com"
BLOG_URL = "https://scouttunisien.com/blog/"
OUTPUT_XLSX = "blog_posts.xlsx"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
    )
}

REQUEST_TIMEOUT = 30
REQUEST_DELAY_SEC = 0.5  # polite delay between requests


def fetch_soup(url: str) -> BeautifulSoup:
    resp = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def norm_text(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()


def first_text(el) -> str:
    if not el:
        return ""
    return norm_text(" ".join(el.stripped_strings))


def find_read_more(article) -> str | None:
    # 1) Elementor "Read more" button
    a = article.select_one("a.elementor-post__read-more")
    if a and a.get("href"):
        return a["href"]

    # 2) Title link
    a = article.select_one(".elementor-post__title a, h2 a, h3 a")
    if a and a.get("href"):
        return a["href"]

    # 3) Any anchor with "voir/see/more/plus"
    for a in article.select("a[href]"):
        t = first_text(a).lower()
        if "voir" in t or "see" in t or "more" in t or "plus" in t:
            return a.get("href")

    # 4) Fallback: first anchor
    a = article.find("a", href=True)
    return a["href"] if a else None


def extract_title(soup: BeautifulSoup) -> str:
    sel = [
        "h1.entry-title",
        ".elementor-heading-title",
        "article h1",
        "header h1",
    ]
    for s in sel:
        el = soup.select_one(s)
        if el:
            return first_text(el)

    # Meta fallback
    og = soup.find("meta", attrs={"property": "og:title"})
    if og and og.get("content"):
        return norm_text(og["content"])

    title_tag = soup.find("title")
    return norm_text(title_tag.get_text()) if title_tag else ""


def extract_date(soup: BeautifulSoup) -> str:
    time_el = soup.select_one("time[datetime]")
    if time_el and time_el.get("datetime"):
        return norm_text(time_el["datetime"])
    if time_el:
        return first_text(time_el)

    el = soup.select_one(".elementor-post-date, .posted-on time, .entry-meta time")
    if el:
        dt = el.get("datetime")
        return norm_text(dt or first_text(el))

    meta_pub = soup.find("meta", attrs={"property": "article:published_time"})
    if meta_pub and meta_pub.get("content"):
        return norm_text(meta_pub["content"])

    meta_date = soup.find("meta", attrs={"name": "date"})
    if meta_date and meta_date.get("content"):
        return norm_text(meta_date["content"])

    return ""


def extract_description(soup: BeautifulSoup) -> str:
    # Prefer main content blocks
    for s in [
        ".entry-content",
        "article .entry-content",
        "main .entry-content",
        ".elementor-widget-text-editor",
        "article .elementor-widget-text-editor",
        "main .elementor-widget-text-editor",
    ]:
        el = soup.select_one(s)
        if el:
            txt = first_text(el)
            if txt:
                return txt

    # Meta description fallback
    meta_desc = soup.find("meta", attrs={"name": "description"})
    if meta_desc and meta_desc.get("content"):
        return norm_text(meta_desc["content"])

    og_desc = soup.find("meta", attrs={"property": "og:description"})
    if og_desc and og_desc.get("content"):
        return norm_text(og_desc["content"])

    # Fallback: overall main/article text
    main = soup.select_one("main, article")
    return first_text(main) if main else ""


def scrape_blog() -> pd.DataFrame:
    results = []
    seen_detail_urls = set()

    page = 1
    while True:
        list_url = BLOG_URL if page == 1 else f"{BLOG_URL}page/{page}/"
        print(f"Scraping list page: {list_url}")

        try:
            list_soup = fetch_soup(list_url)
        except requests.HTTPError as e:
            # Stop when we hit a 404 (no more pages)
            status = getattr(e.response, "status_code", None)
            print(f"Stopping at page {page}: HTTP error {status}")
            break
        except Exception as e:
            print(f"Error fetching {list_url}: {e}")
            break

        # Try Elementor posts container first, then generic articles
        articles = list_soup.select(".elementor-posts-container article")
        if not articles:
            articles = list_soup.select("article")

        # If a paginated page has no articles, we assume no more pages
        if page > 1 and not articles:
            print(f"No articles on page {page}; stopping.")
            break

        print(f"Found {len(articles)} articles on page {page}")
        for art in articles:
            href = find_read_more(art)
            if not href:
                continue
            detail_url = urljoin(BASE_URL, href)
            if detail_url in seen_detail_urls:
                continue
            seen_detail_urls.add(detail_url)

            # Fetch detail page
            try:
                time.sleep(REQUEST_DELAY_SEC)
                detail_soup = fetch_soup(detail_url)
            except Exception as e:
                print(f"Failed to fetch detail {detail_url}: {e}")
                continue

            title = extract_title(detail_soup)
            date = extract_date(detail_soup)
            desc = extract_description(detail_soup)

            results.append(
                {
                    "name": title,
                    "date": date,
                    "description": desc,
                    "url": detail_url,  # kept internally, not written to Excel
                }
            )

            # Be polite between detail fetches
            time.sleep(REQUEST_DELAY_SEC)

        # Next page
        page += 1
        # Optional: safety upper bound to avoid infinite loops; set to None to disable
        if page > 200:
            print("Reached page limit safeguard (200); stopping.")
            break

    return pd.DataFrame(results)


def save_to_excel(df: pd.DataFrame, path: str = OUTPUT_XLSX):
    if df.empty:
        print("No blog posts found; nothing to save.")
        return
    out = df[["name", "date", "description"]]  # only requested columns
    # Requires 'openpyxl' installed
    out.to_excel(path, index=False)
    print(f"Saved {len(out)} rows to {path}")


if __name__ == "__main__":
    df = scrape_blog()
    print("Total rows:", len(df))
    # Preview first few
    print(df[["name", "date"]].head(10))
    save_to_excel(df, OUTPUT_XLSX)

Scraping list page: https://scouttunisien.com/blog/
Found 10 articles on page 1
Scraping list page: https://scouttunisien.com/blog/page/2/
Found 10 articles on page 2
Scraping list page: https://scouttunisien.com/blog/page/3/
Found 10 articles on page 3
Scraping list page: https://scouttunisien.com/blog/page/4/
Found 10 articles on page 4
Scraping list page: https://scouttunisien.com/blog/page/5/
Found 10 articles on page 5
Scraping list page: https://scouttunisien.com/blog/page/6/
Found 10 articles on page 6
Scraping list page: https://scouttunisien.com/blog/page/7/
Found 2 articles on page 7
Scraping list page: https://scouttunisien.com/blog/page/8/
Stopping at page 8: HTTP error 404
Total rows: 62
                                                name  \
0  Ce que le scout tunisien ont créé à Djerba… es...   
1                       Cِamp d’hiver Pour régiments   
2  Réunion des chefs des Régiments à Sfax le 25 d...   
3  Les formations nationales de décembre 2025 à l...   
4  Campagn